<a href="https://colab.research.google.com/github/logankim0913/EE_467_Final_Project/blob/standardScaler/phase1_standardscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

file_path = os.path.join("DictionaryBruteForce.pcap.csv")
#read the csv files and print the first 10 rows of each
bruteforcedata = pd.read_csv(file_path)
bruteforcedata.head(10)

spoofdata = pd.read_csv("DNS_Spoofing.pcap.csv")
spoofdata.head(10)

benigndata = pd.read_csv("BenignTraffic.pcap.csv", on_bad_lines='skip')
benigndata.head(10)

# #Preview statistics of the data
print("Brute Force Data Statistics:")
print(bruteforcedata.describe())
print("\nDNS Spoofing Data Statistics:")
print(spoofdata.describe())
print("\nBenign Traffic Data Statistics:")
print(benigndata.describe())

Brute Force Data Statistics:
       Header_Length  Protocol Type  Time_To_Live           Rate  \
count   13064.000000   13064.000000  13064.000000   13064.000000   
mean       23.883762       7.632961     93.120070    1732.280240   
std         7.323148       4.064012     34.447147   24243.212110   
min         0.800000       0.000000     15.600000       0.000040   
25%        18.800000       6.000000     64.000000      52.411901   
50%        24.800000       6.000000     83.100000     106.396493   
75%        30.400000       6.000000    112.500000     249.954800   
max        44.800000      17.000000    247.000000  762600.727273   

       fin_flag_number  syn_flag_number  rst_flag_number  psh_flag_number  \
count     13064.000000     13064.000000     13064.000000     13064.000000   
mean          0.026607         0.046249         0.010257         0.251944   
std           0.075118         0.089139         0.036774         0.195413   
min           0.000000         0.000000         0.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# handle infinite values before scaling
# drop rows with infinity
benigndata_cleaned = benigndata.replace([np.inf, -np.inf], np.nan).dropna()
spoofdata_cleaned = spoofdata.replace([np.inf, -np.inf], np.nan).dropna()

# Explicitly convert all columns to numeric, coercing errors, then drop rows with NaNs
for col in benigndata_cleaned.columns:
    benigndata_cleaned[col] = pd.to_numeric(benigndata_cleaned[col], errors='coerce')
benigndata_cleaned = benigndata_cleaned.dropna()

for col in spoofdata_cleaned.columns:
    spoofdata_cleaned[col] = pd.to_numeric(spoofdata_cleaned[col], errors='coerce')
spoofdata_cleaned = spoofdata_cleaned.dropna()

#obtaining features as a list for each chosen class
bfd_scaled = bruteforcedata.copy()
categorical_col_bfd = []
for col in bfd_scaled.columns:
  categorical_col_bfd.append(col)
print(categorical_col_bfd)
benign_scaled = benigndata_cleaned.copy() #no inf
categorical_col_benign = []
for col in benign_scaled.columns:
  categorical_col_benign.append(col)

spoof_scaled = spoofdata_cleaned.copy() #no inf
categorical_col_spoof = []
for col in spoof_scaled.columns:
  categorical_col_spoof.append(col)

#Standard scaling (no outlier removal yet)
bfd_feat_std = pd.DataFrame(StandardScaler().fit_transform(bruteforcedata[categorical_col_bfd]), columns=categorical_col_bfd)
benign_feat_std = pd.DataFrame(StandardScaler().fit_transform(benign_scaled[categorical_col_benign]), columns=categorical_col_benign)
spoof_feat_std = pd.DataFrame(StandardScaler().fit_transform(spoof_scaled[categorical_col_spoof]), columns=categorical_col_spoof)

# Outlier removal logic
remove_outliers = True
outlier_threshold = 5

#Dictionary Brute Force Data
if remove_outliers:
    outlier_indices_bfd = set()
    for col_name in bfd_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = bfd_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indices with `outlier_indices_bfd`
        outlier_indices_bfd.update(out_indices.tolist())
    #Remove outliers
    bfd_feat_std = bfd_feat_std.drop(index=list(outlier_indices_bfd))

    # Benign Data
    outlier_indices_benign = set()
    for col_name in benign_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = benign_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        # Merge these indices with `outlier_indices_benign`
        outlier_indices_benign.update(out_indices.tolist())
    #Remove outliers
    benign_feat_std = benign_feat_std.drop(index=list(outlier_indices_benign))

    # DNS Spoofing Data
    outlier_indices_spoof = set()
    for col_name in spoof_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = spoof_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indives with 'outlier_indices_spoof"
        outlier_indices_spoof.update(out_indices.tolist())
    #Remove outliers
    spoof_feat_std = spoof_feat_std.drop(index=list(outlier_indices_spoof))

print("Brute Force Scaled Data (without outliers):")
print(bfd_feat_std.head())
print("\nBenign Scaled Data (without outliers):")
print(benign_feat_std.head())
print("\nSpoof Scaled Data (without outliers):")
print(spoof_feat_std.head())

['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']
Brute Force Scaled Data (without outliers):
   Header_Length  Protocol Type  Time_To_Live      Rate  fin_flag_number  \
0       0.452861      -0.401825      0.701970 -0.070212         2.308350   
1       0.835224      -0.401825      0.150379 -0.070211        -0.354221   
2      -0.584984      -0.401825     -0.232831 -0.066449        -0.354221   
3       0.288990      -0.401825      0.710679 -0.070138        -0.354221   
4      -0.967348       2.304963      1.070665 -0.067000        -0.354221   

   syn_flag_number  rst_flag_number  psh_flag_number  ack_fl

In [ ]:
#NOT CURRENTLY APPLIED TO PHASE 1
#Just a start to potential Robust Scaling if eventually needed
from sklearn.preprocessing import StandardScaler, RobustScaler

# Copy original dataset
bruteforce_scaled = bruteforcedata.copy()

# Convert transaction time from seconds to hour-of-day (0-23) then scale with `StandardScaler`
time_in_hours = bruteforcedata["Time_To_Live"] / (60 * 60)
hour_of_day = time_in_hours % 24
time_scaled = StandardScaler().fit_transform(hour_of_day.values.reshape(-1, 1))
bruteforce_scaled["Time_To_Live"] = time_scaled

# Rescale and override transaction amount column with `RobustScaler`
amount_scaled = RobustScaler().fit_transform(bruteforcedata["Tot size"].values.reshape(-1, 1))
bruteforce_scaled["Tot size"] = amount_scaled

# Review scaling results
print("Scaled Brute Force Data:")
bruteforce_scaled[["Tot size", "Time_To_Live"]].head(5)

Scaled Brute Force Data:


,Tot size,Time_To_Live
0,-0.262547,0.701970
1,-0.270037,0.150379
2,-0.151685,-0.232831
3,-0.045318,0.710679
4,2.498502,1.070665


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def clean_df(df):
    #Replaces inf/-inf with NaN and drops those selected rows
    return df.replace([np.inf, -np.inf], np.nan).dropna()

#scaling only on benign data
bfd = pd.read_csv("DictionaryBruteForce.pcap.csv")
spoof = pd.read_csv("DNS_Spoofing.pcap.csv")
benign = pd.read_csv("BenignTraffic.pcap.csv", on_bad_lines='skip')

#all features extract
bfd_copy = bfd.copy()
categorical_col = []
for col in bfd_copy.columns:
  categorical_col.append(col)

#clean data so no inf
bfd = clean_df(bfd)
spoof = clean_df(spoof)
benign = clean_df(benign)

# Explicitly convert all columns to numeric, coercing errors, then drop rows with NaNs
for col in bfd.columns:
    bfd[col] = pd.to_numeric(bfd[col], errors='coerce')
bfd = bfd.dropna()

for col in spoof.columns:
    spoof[col] = pd.to_numeric(spoof[col], errors='coerce')
spoof = spoof.dropna()

for col in benign.columns:
    benign[col] = pd.to_numeric(benign[col], errors='coerce')
benign = benign.dropna()

#Test split
RANDOM_SEED = 42

#benign data split so train = 70%, val and test both 15%
benign_train, benign_tmp = train_test_split(
    benign, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
)
benign_val, benign_test = train_test_split(
    benign_tmp, test_size=0.50, random_state=RANDOM_SEED, shuffle=True
)

syn_data_columns = benign_train.columns # Assuming benign_train has all the relevant columns

scaler = StandardScaler()
#training set for benign
X_train = scaler.fit_transform(benign_train[syn_data_columns].values)
#validation set for benign
X_val_benign = scaler.transform(benign_val[syn_data_columns].values)

# test set includes benign_test + all attacks
test = pd.concat(
    [
        benign_test.assign(label=0),   # 0 = benign
        bfd.assign(label=1),           # 1 = attack
        spoof.assign(label=1),
    ],
    ignore_index=True
)

#test set has both benign and attack
X_test = scaler.transform(test[categorical_col].values)
#labels for test set
y_test = test["label"].values

print("Shapes:")
print("X_train (benign):", X_train.shape)
print("X_val_benign:", X_val_benign.shape)
print("X_test (mixed):", X_test.shape, "y_test:", y_test.shape)

print("\nTest class balance:")
print(pd.Series(y_test).value_counts())

#my output was 191957 for attack (1) and 54352 for benign (0)
#HIGHLY IMBALANCED (much more attack)


Shapes:
X_train (benign): (63788, 39)
X_val_benign: (13669, 39)
X_test (mixed): (128472, 39) y_test: (128472,)

Test class balance:
1    114803
0     13669
Name: count, dtype: int64


In [ ]:
bfd_train, bfd_test = train_test_split(
    bfd, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
)

spoof_train, spoof_test = train_test_split(
    spoof, test_size=0.30, random_state=RANDOM_SEED, shuffle=True
)

In [ ]:
train_full = pd.concat(
    [
        benign_train.assign(label=0),
        bfd_train.assign(label=1),
        spoof_train.assign(label=1),
    ]
)

test = pd.concat(
    [
        benign_test.assign(label=0),
        bfd_test.assign(label=1),
        spoof_test.assign(label=1),
    ],
    ignore_index=True
)

In [ ]:
X_train_full = scaler.transform(train_full[categorical_col].values)
y_train_full = train_full["label"].values

In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

In [ ]:
#Built autoencoder
def build_autoencoder(input_dim):

    inputs = keras.Input(shape=(input_dim,))

    # Encoder
    x = layers.Dense(32, activation='relu')(inputs)
    x = layers.Dense(16, activation='relu')(x)
    latent = layers.Dense(8, activation='relu')(x)

    # Decoder
    x = layers.Dense(16, activation='relu')(latent)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(input_dim)(x)

    model = keras.Model(inputs, outputs)

    encoder = keras.Model(inputs, latent)

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model, encoder

In [ ]:
#instantiate model
ae, encoder = build_autoencoder(X_train.shape[1])

#parameter count
def count_parameters(model):
    return np.sum([np.prod(v.shape) for v in model.trainable_weights])

print("Trainable Parameters:", count_parameters(ae))

Trainable Parameters: 3919


In [ ]:
#Train autoencoder
history = ae.fit(
    X_train,
    X_train,
    epochs=40,
    batch_size=256,
    validation_data=(X_val_benign, X_val_benign),
    verbose=1
)

Epoch 1/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.7443 - val_loss: 0.3871
Epoch 2/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3585 - val_loss: 0.2535
Epoch 3/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2323 - val_loss: 0.1800
Epoch 4/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1729 - val_loss: 0.1504
Epoch 5/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1548 - val_loss: 0.1352
Epoch 6/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1350 - val_loss: 0.1253
Epoch 7/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1272 - val_loss: 0.1153
Epoch 8/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1175 - val_loss: 0.1068
Epoch 9/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1093 - val_loss: 0.1024
Epoch 10/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1025 - val_loss: 0.0940
Epoch 11/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0921 - val_loss: 0.0853
Epoch 12/40
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

In [ ]:
#Reconstruction error
def reconstruction_error(model, X):
    reconstructed = model.predict(X, verbose=0)
    error = np.mean((X - reconstructed) ** 2, axis=1)
    return error

In [ ]:
#Threshold at 0.5% FPR
def compute_threshold(val_errors, target_fpr=0.005):
    return np.percentile(val_errors, 100 * (1 - target_fpr))

val_errors = reconstruction_error(ae, X_val_benign)
threshold = compute_threshold(val_errors, 0.005)

print("0.5% FPR Threshold:", threshold)

0.5% FPR Threshold: 0.5434838472049065


In [ ]:
#AE detection on test set
def ae_predict(model, X, threshold):
    errors = reconstruction_error(model, X)
    preds = (errors > threshold).astype(int)
    return preds, errors

ae_preds, ae_errors = ae_predict(ae, X_test, threshold)

In [ ]:
def count_lr_params(model):
  # Coefficients (one per feature per class, but binary only needs one set)
  n_coefficients = logreg.coef_.size

  # Intercepts (one per class, binary only needs one)
  n_intercepts = logreg.intercept_.size

  total_params = n_coefficients + n_intercepts
  print(f"Total parameters: {total_params}")

In [ ]:
#Logistic Regression baseline
logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    n_jobs=-1
)

logreg.fit(X_train_full, y_train_full)
lr_preds = logreg.predict(X_test)
count_lr_params(logreg)

Total parameters: 40


In [ ]:
#Autoencoder Evaluation
from sklearn.metrics import classification_report
tn, fp, fn, tp = confusion_matrix(y_test, ae_preds).ravel()

ae_recall = tp / (tp + fn)
ae_fpr = fp / (fp + tn)

#Logistic Regression Evaluation
tn, fp, fn, tp = confusion_matrix(y_test, lr_preds).ravel()

lr_recall = tp / (tp + fn)
lr_fpr = fp / (fp + tn)

print("AE Recall @ 0.5% FPR:", ae_recall)
print("AE Actual FPR:", ae_fpr, "\n")

print("AE Classification Report: \n\n", classification_report(y_test, ae_preds))
print("\nLogistic Regression Classification Report: \n\n", classification_report(y_test, lr_preds))

print("LogReg Recall:", lr_recall)
print("LogReg FPR:", lr_fpr, "\n")

AE Recall @ 0.5% FPR: 0.04657543792409606
AE Actual FPR: 0.004243177994001025 

AE Classification Report: 

               precision    recall  f1-score   support

           0       0.11      1.00      0.20     13669
           1       0.99      0.05      0.09    114803

    accuracy                           0.15    128472
   macro avg       0.55      0.52      0.14    128472
weighted avg       0.90      0.15      0.10    128472


Logistic Regression Classification Report: 

               precision    recall  f1-score   support

           0       0.28      0.76      0.41     13669
           1       0.96      0.77      0.86    114803

    accuracy                           0.77    128472
   macro avg       0.62      0.76      0.64    128472
weighted avg       0.89      0.77      0.81    128472

LogReg Recall: 0.7728543679172147
LogReg FPR: 0.24332431048357597 



In [ ]:
def distance_scoring(encoder, kmeans, X):
  latent = encoder.predict(X, verbose=0)
  distances = kmeans.transform(latent)
  min_distances = distances.min(axis=1)
  return min_distances

def ae_kmeans_predict(encoder, kmeans, X, threshold):
    distances = distance_scoring(encoder, kmeans, X)
    preds = (distances > threshold).astype(int)
    return preds, distances

In [ ]:
from sklearn.cluster import MiniBatchKMeans, KMeans
# Now latent space is meaningful
latent_train = encoder.predict(X_train)
kmeans = KMeans(random_state=0, max_iter=500)
kmeans.fit(latent_train)

distances = distance_scoring(encoder, kmeans, X_val_benign)
scoring_threshold = np.percentile(distances, 99.5)

ae_km_preds, ae_km_distances = ae_kmeans_predict(encoder, kmeans, X_test, scoring_threshold)

1994/1994 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, ae_km_preds).ravel()

ae_k_recall = tp / (tp + fn)
ae_k_fpr = fp / (fp + tn)

print("AE K-means Recall @ 0.5% FPR:", ae_k_recall)
print("AE K-Means Actual FPR:", ae_k_fpr, "\n")

print("AE Classification Report: \n\n", classification_report(y_test, ae_km_preds))

AE K-means Recall @ 0.5% FPR: 0.03995540186232067
AE K-Means Actual FPR: 0.004901602165483941 

AE Classification Report: 

               precision    recall  f1-score   support

           0       0.11      1.00      0.20     13669
           1       0.99      0.04      0.08    114803

    accuracy                           0.14    128472
   macro avg       0.55      0.52      0.14    128472
weighted avg       0.89      0.14      0.09    128472



In [ ]:
def count_rf_params(model):
    total_nodes = sum(tree.tree_.node_count for tree in model.estimators_)

    print(f"Number of trees: {len(model.estimators_)}")
    print(f"Total nodes: {total_nodes}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
N_ENSEMBLE_CPUS = max(os.cpu_count()//2, 1)
rf_15_model = RandomForestClassifier(n_estimators=15, n_jobs=N_ENSEMBLE_CPUS).fit(X_train_full, y_train_full)
rf_15_preds = rf_15_model.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, rf_15_preds).ravel()

rf_recall = tp / (tp + fn)
rf_fpr = fp / (fp + tn)

print("\nRF Classification Report: \n\n", classification_report(y_test, rf_15_preds))

print("RF 15 Recall:", rf_recall)
print("RF 15 FPR:", rf_fpr, "\n")

print("Random Forest Classifier Model Features:")
count_rf_params(rf_15_model)


RF Classification Report: 

               precision    recall  f1-score   support

           0       0.80      0.92      0.86     13669
           1       0.99      0.97      0.98    114803

    accuracy                           0.97    128472
   macro avg       0.90      0.95      0.92    128472
weighted avg       0.97      0.97      0.97    128472

RF 15 Recall: 0.9723526388683222
RF 15 FPR: 0.075426146755432 

Random Forest Classifier Model Features:
Number of trees: 15
Total nodes: 316117
